# Object Detection Task — Laptop / Phone / Glasses in the Classroom (YOLOv8 OBB)

**Pipeline (as in the task sheet):** Problem selection → Image collection → Annotation in Roboflow → Dataset preparation in Roboflow → Train YOLOv8 Oriented Bounding Boxes on a Colab GPU with different hyperparameters → Evaluation & testing → Gradio deployment (optional) → Presentation summary.

**Before running:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

## Step 1: Problem Selection
- **Task:** detect the personal devices and accessories students leave on classroom desks.
- **Classes (3 = the maximum allowed):** **Glasses**, **Laptop**, **Phone**.
- The objects are clearly visible, common and realistic. Many lie at an angle on the desks, which is why we use **oriented (rotated) bounding boxes**.

## Step 2: Image Collection
- Real photos taken with our **phone cameras** in our classroom by three team members, uploaded to **Roboflow**.
- Current Roboflow version (v1): **43 images** (target range: 30–60). Seven of one member's 20 images are not uploaded yet.

## Step 3: Annotation in Roboflow
- Every image was annotated with polygons in the Roboflow editor, and labels were reviewed by the team.
- During review we merged duplicate class names (laptop/Laptop, phone/Phone, glasses/Glasses) and removed an empty class, leaving exactly 3 classes.
- **146 labelled objects:** Laptop 71, Phone 61, Glasses 14.

## Step 4: Dataset Preparation (Roboflow)
- **Train / Validation / Test split:** 30 / 10 / 3 images. Near-identical burst photos were kept in the same split to limit leakage.
- **Preprocessing:** Auto-Orient, resize to 640×640 (fit with black edges).
- **Light augmentation:** flip, brightness and blur. In this notebook flip and brightness are applied on the fly by YOLO during training (fliplr, hsv_v), and YOLO's built-in albumentations add a small amount of blur.
- **Export:** YOLOv8 Oriented Bounding Boxes (.zip).

In [ ]:
# Setup
!pip install -q ultralytics gradio
import os, shutil, zipfile, math, yaml, tempfile
from pathlib import Path
from collections import Counter
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from ultralytics.utils.metrics import batch_probiou
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> Runtime > Change runtime type > GPU")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

### Load the Roboflow export (YOLOv8 Oriented Bounding Boxes .zip)
If the dataset is not already in this Colab session, a file picker appears: choose the zip exported from Roboflow.

In [ ]:
RAW = Path("/content/dataset_raw")
if not (RAW / "train" / "images").exists():
    from google.colab import files
    print("Upload the Roboflow export (YOLOv8 Oriented Bounding Boxes .zip)")
    up = files.upload()
    shutil.rmtree(RAW, ignore_errors=True)
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(RAW)
    inner = [p for p in RAW.iterdir() if p.is_dir() and not p.name.startswith("__")]
    if not (RAW / "train").exists() and len(inner) == 1:
        RAW = inner[0]

CLASS_NAMES = yaml.safe_load(open(RAW / "data.yaml"))["names"]
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES)]
print("Dataset folder:", RAW)
print("Classes:", CLASS_NAMES)

In [ ]:
# Clean copy of the dataset: every label becomes a 4-corner oriented box clamped inside the image
DATA = Path("/content/work/dataset_obb")
shutil.rmtree(DATA, ignore_errors=True)
SPLITS = ["train", "valid", "test"]
IMG_EXT = {".jpg", ".jpeg", ".png"}
clamped, label_rows = 0, []

for split in SPLITS:
    (DATA / split / "images").mkdir(parents=True)
    (DATA / split / "labels").mkdir(parents=True)
    for img in sorted((RAW / split / "images").iterdir()):
        if img.suffix.lower() not in IMG_EXT:
            continue
        shutil.copy(img, DATA / split / "images" / img.name)
        src, out = RAW / split / "labels" / (img.stem + ".txt"), []
        if src.exists():
            for line in open(src):
                p = line.split()
                if len(p) < 7:
                    continue
                cls = int(float(p[0]))
                pts = np.array(p[1:], dtype=np.float32).reshape(-1, 2)
                if (pts < 0).any() or (pts > 1).any():
                    clamped += 1
                box = cv2.boxPoints(cv2.minAreaRect(np.clip(pts, 0, 1) * 1000)) / 1000   # images are square (640x640)
                out.append(f"{cls} " + " ".join(f"{v:.6f}" for v in np.clip(box, 0, 1).flatten()))
                label_rows.append({"split": split, "class": CLASS_NAMES[cls]})
        (DATA / split / "labels" / (img.stem + ".txt")).write_text("\n".join(out) + ("\n" if out else ""))

DATA_YAML = Path("/content/work/data_obb.yaml")
yaml.safe_dump({"path": str(DATA), "train": "train/images", "val": "valid/images", "test": "test/images",
                "names": dict(enumerate(CLASS_NAMES))}, open(DATA_YAML, "w"), sort_keys=False)

N_IMAGES = {s: len(list((DATA / s / "images").iterdir())) for s in SPLITS}
N_OBJECTS = len(label_rows)
print("Images per split:", N_IMAGES, "| total images:", sum(N_IMAGES.values()), "| total objects:", N_OBJECTS)
print("Boxes clamped to the image border:", clamped)
lr = pd.DataFrame(label_rows)
display(pd.crosstab(lr["class"], lr["split"], margins=True, margins_name="total")[SPLITS + ["total"]])

### All dataset images with their ground-truth labels
Colours: **Glasses = orange**, **Laptop = blue**, **Phone = red**.

In [ ]:
COLORS = {0: (255, 140, 0), 1: (0, 170, 255), 2: (255, 40, 40)}   # RGB for class 0, 1, 2

def load_rgb(p):
    return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

def read_gt(img_path):
    lbl = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
    gts = []
    if lbl.exists():
        for line in open(lbl):
            p = line.split()
            if len(p) == 9:
                gts.append((int(p[0]), np.array(p[1:], dtype=np.float32).reshape(4, 2)))
    return gts

def draw(img, items, normalized=True):
    """items: (cls, 4x2 points) or (cls, 4x2 points, confidence)."""
    img = img.copy(); h, w = img.shape[:2]
    for cls, pts, *conf in items:
        pts = (pts * [w, h] if normalized else pts).astype(np.int32)
        col = COLORS.get(cls, (0, 255, 0))
        cv2.polylines(img, [pts], True, col, 3)
        txt = CLASS_NAMES[cls] + (f" {conf[0]:.2f}" if conf else "")
        org = (int(pts[:, 0].min()), int(max(pts[:, 1].min() - 6, 16)))
        cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 4)
        cv2.putText(img, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
    return img

def show_grid(images, titles, cols=4, size=4.5):
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()

def short(name, n=26):
    return name if len(name) <= n else name[:n] + "…"

for split in SPLITS:
    paths = sorted((DATA / split / "images").iterdir())
    print(f"\n{split.upper()}: {len(paths)} images (ground truth)")
    show_grid([draw(load_rgb(p), read_gt(p)) for p in paths],
              [f"{short(p.name)} | {len(read_gt(p))} objects" for p in paths])

## Step 5: Train the YOLO model (YOLOv8 OBB on the Colab GPU)
Three experiments with **different hyperparameters**:

| Experiment | Model | Optimizer / learning rate | Purpose |
|---|---|---|---|
| A | yolov8n-obb (nano) | auto (AdamW) | baseline |
| B | yolov8n-obb (nano) | SGD, lr0 = 0.005 | effect of optimizer / learning rate |
| C | yolov8s-obb (small) | auto (AdamW) | effect of a bigger model |

All runs: 640 px, batch 8, up to 100 epochs with early stopping (patience 30), flip + brightness/colour augmentation.

In [ ]:
COMMON = dict(data=str(DATA_YAML), imgsz=640, batch=8, epochs=100, patience=30, device=DEVICE, seed=42,
              plots=True, project="/content/work/runs_obb", exist_ok=True,
              fliplr=0.5, flipud=0.0, degrees=15,          # flip + small rotations
              hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,           # colour + brightness
              mosaic=1.0, mixup=0.0)
EXPERIMENTS = {
    "A_nano_default": dict(model="yolov8n-obb.pt"),
    "B_nano_SGD_lr0.005": dict(model="yolov8n-obb.pt", optimizer="SGD", lr0=0.005),
    "C_small_default": dict(model="yolov8s-obb.pt"),
}
RUNS = {}
for name, cfg in EXPERIMENTS.items():
    cfg = dict(cfg); weights = cfg.pop("model")
    print(f"\n#################### {name} ({weights}) ####################")
    model = YOLO(weights)
    model.train(name=name, **COMMON, **cfg)
    RUNS[name] = Path(model.trainer.save_dir)
print({k: str(v) for k, v in RUNS.items()})

### Training curves and all plots of every experiment
Loss and metric curves, label statistics, confusion matrices, precision / recall / F1 / PR curves, training batches and validation predictions.

In [ ]:
for name, run in RUNS.items():
    order = ["results.png", "labels.jpg", "confusion_matrix.png", "confusion_matrix_normalized.png"]
    files = [f for f in order if (run / f).exists()]
    files += sorted(p.name for p in run.glob("*curve*.png"))
    files += sorted(p.name for p in run.glob("train_batch*.jpg")) + sorted(p.name for p in run.glob("val_batch*.jpg"))
    print(f"\n==================== {name}: {len(files)} plots ====================")
    for f in files:
        plt.figure(figsize=(16, 9)); plt.imshow(load_rgb(run / f)); plt.title(f"{name} | {f}"); plt.axis("off"); plt.show()

## Step 6: Evaluation & Testing
Precision, recall, mAP@50 and mAP@50–95 for every experiment on the validation and test sets, overall and per class. The best experiment is chosen by **validation mAP@50–95**.

In [ ]:
metric_rows = []
for name, run in RUNS.items():
    best = YOLO(run / "weights" / "best.pt")
    for split in ["val", "test"]:
        r = best.val(data=str(DATA_YAML), split=split, plots=False, verbose=False)
        metric_rows.append(dict(experiment=name, split=split, cls="ALL", P=r.box.mp, R=r.box.mr,
                                mAP50=r.box.map50, mAP50_95=r.box.map))
        for i, c in enumerate(r.box.ap_class_index):
            metric_rows.append(dict(experiment=name, split=split, cls=CLASS_NAMES[c], P=r.box.p[i], R=r.box.r[i],
                                    mAP50=r.box.ap50[i], mAP50_95=r.box.ap[i]))
METRICS = pd.DataFrame(metric_rows).round(3)

print("Overall metrics per experiment")
display(METRICS[METRICS.cls == "ALL"].pivot(index="experiment", columns="split", values=["P", "R", "mAP50", "mAP50_95"]))
print("Per-class validation metrics")
display(METRICS[(METRICS.split == "val") & (METRICS.cls != "ALL")].set_index(["cls", "experiment"])[["P", "R", "mAP50", "mAP50_95"]])

ov = METRICS[(METRICS.cls == "ALL") & (METRICS.split == "val")].set_index("experiment")
ov[["P", "R", "mAP50", "mAP50_95"]].plot.bar(figsize=(10, 4), rot=0, title="Validation metrics by experiment"); plt.ylim(0, 1); plt.show()

BEST_NAME = ov["mAP50_95"].idxmax()
BEST = YOLO(RUNS[BEST_NAME] / "weights" / "best.pt")
print("Best experiment (highest validation mAP50-95):", BEST_NAME)

### Predictions of the best model on every image
Left: ground truth. Right: prediction (confidence ≥ 0.25). Validation and test images were **never seen in training**. Training images are shown too, for completeness. Matching uses IoU ≥ 0.5 (rotated-box probIoU) between objects of the same class.

In [ ]:
CONF, IOU_T = 0.25, 0.5

def xywhr(pts_px):
    (cx, cy), (w, h), a = cv2.minAreaRect(pts_px.astype(np.float32))
    return [cx, cy, w, h, np.deg2rad(a)]

def predict(p):
    r = BEST.predict(str(p), conf=CONF, verbose=False)[0]
    if r.obb is None or len(r.obb) == 0:
        return []
    return [(int(c), poly, float(s)) for c, s, poly in
            zip(r.obb.cls.cpu().numpy(), r.obb.conf.cpu().numpy(), r.obb.xyxyxyxy.cpu().numpy())]

def match(gts, preds, w, h):
    used, detected, missed = set(), [], []
    for gc, gpts in gts:
        best_j, best_iou = None, IOU_T
        for j, (pc, ppoly, _) in enumerate(preds):
            if j in used or pc != gc:
                continue
            iou = batch_probiou(torch.tensor([xywhr(gpts * [w, h])], dtype=torch.float32),
                                torch.tensor([xywhr(ppoly)], dtype=torch.float32)).item()
            if iou >= best_iou:
                best_j, best_iou = j, iou
        (missed if best_j is None else detected).append(gc)
        if best_j is not None:
            used.add(best_j)
    false_alarms = [pc for j, (pc, _, _) in enumerate(preds) if j not in used]
    return detected, missed, false_alarms

EVAL = {c: Counter() for c in range(len(CLASS_NAMES))}
MISSED_LIST = []
for split in ["valid", "test", "train"]:
    paths = sorted((DATA / split / "images").iterdir())
    panels, titles = [], []
    for p in paths:
        img = load_rgb(p); h, w = img.shape[:2]
        gts, preds = read_gt(p), predict(p)
        det, mis, fa = match(gts, preds, w, h)
        if split != "train":
            for c in det: EVAL[c]["detected"] += 1
            for c in mis: EVAL[c]["missed"] += 1
            for c in fa: EVAL[c]["false alarms"] += 1
            if mis:
                MISSED_LIST.append((split, p.name, ", ".join(CLASS_NAMES[c] for c in mis)))
        panels += [draw(img, gts), draw(img, [(c, poly, s) for c, poly, s in preds], normalized=False)]
        titles += [f"GT | {short(p.name)}", f"PRED | found {len(det)}, missed {len(mis)}, false {len(fa)}"]
    print(f"\n{split.upper()} ({len(paths)} images){'  <- seen during training' if split == 'train' else ''}")
    show_grid(panels, titles, cols=4, size=4.5)

In [ ]:
# Which objects are detected well, and which are missed? (validation + test images)
ev = pd.DataFrame({CLASS_NAMES[c]: {"objects": s["detected"] + s["missed"], "detected": s["detected"],
                                     "missed": s["missed"], "false alarms": s["false alarms"]} for c, s in EVAL.items()}).T
ev["detection rate"] = (ev["detected"] / ev["objects"].replace(0, np.nan)).round(2)
display(ev)

for cls, row in ev.sort_values("detection rate", ascending=False).iterrows():
    if row["objects"] == 0:
        print(f"- {cls}: no examples in the validation/test images")
        continue
    verdict = "detected well" if row["detection rate"] >= 0.7 else ("partly detected" if row["detection rate"] >= 0.4 else "often missed")
    print(f"- {cls}: {verdict} ({int(row['detected'])}/{int(row['objects'])} found, {int(row['false alarms'])} false alarms)")

print("\nImages with missed objects:")
for split, name, what in MISSED_LIST:
    print(f"  [{split}] {name}: missed {what}")

In [ ]:
# Save the best model (set DOWNLOAD = True to download best.pt to your computer)
DOWNLOAD = False
shutil.copy(RUNS[BEST_NAME] / "weights" / "best.pt", "/content/best_obb.pt")
print("Saved /content/best_obb.pt from", BEST_NAME)
if DOWNLOAD:
    from google.colab import files
    files.download("/content/best_obb.pt")